# DS605: Fundamentals of Machine Learning — Lab 5

## Machine Learning with Scikit-learn and From Scratch

1. **Part A:** Scikit-learn implementation
2. **Part B:** NumPy/Pandas implementation from scratch
3. **Part C:** Fair comparison and manual optimization


## 1. Imports and file path


In [8]:
from pathlib import Path
import time
import numpy as np
import pandas as pd

DATA_PATH = Path("garments_worker_productivity.csv")
SPLIT_PATH = Path("split_indices.csv")

RANDOM_STATE = 42
TEST_SIZE = 0.20

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Put garments_worker_productivity.csv "
        "in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns exactly as read:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
display(df.dtypes)


Dataset shape: (2397, 15)

Columns exactly as read:
['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'actual_productivity']

First 5 rows:


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.8,26.16,1108,7080,98,0,0,0,59,0.940725424
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0,0,0,8,0.8865
2,1/1/2015,Quarter1,sweing,Thursday,11,0.8,11.41,968,3660,50,0,0,0,30.5,0.800570492
3,1/1/2015,Quarter1,sweing,Thursday,12,0.8,11.41,968,3660,50,0,0,0,30.5,0.800570492
4,1/1/2015,Quarter1,sweing,Thursday,6,0.8,25.9,1170,1920,50,0,0,0,56,0.800381944



Data types:


date                     str
quarter                  str
department               str
day                      str
team                     str
targeted_productivity    str
smv                      str
wip                      str
over_time                str
incentive                str
idle_time                str
idle_men                 str
no_of_style_change       str
no_of_workers            str
actual_productivity      str
dtype: object

In [9]:
# Remove accidental spaces around column names.
df.columns = df.columns.astype(str).str.strip()

print("Cleaned columns:")
print(df.columns.tolist())

required_columns = {
    "actual_productivity",
    "targeted_productivity",
    "department",
    "quarter",
    "day",
}

missing_required = required_columns - set(df.columns)

if missing_required:
    raise ValueError(
        f"Missing required columns: {sorted(missing_required)}\n"
        f"Columns found: {df.columns.tolist()}"
    )

for col in ["department", "quarter", "day"]:
    df[col] = df[col].astype(str).str.strip()

print("\nDepartment values:")
print(df["department"].value_counts(dropna=False))

print("\nQuarter values:")
print(df["quarter"].value_counts(dropna=False))

print("\nDay values:")
print(df["day"].value_counts(dropna=False))

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing"))


Cleaned columns:
['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'actual_productivity']

Department values:
department
sweing        1382
finishing     1012
NaN              2
department       1
Name: count, dtype: int64

Quarter values:
quarter
Quarter1    720
Quarter2    670
Quarter4    496
Quarter3    420
Quarter5     88
NaN           2
quarter       1
Name: count, dtype: int64

Day values:
day
Wednesday    416
Sunday       406
Tuesday      402
Thursday     398
Monday       398
Saturday     374
NaN            2
day            1
Name: count, dtype: int64

Missing values:


,Missing
wip,1014
department,2
quarter,2
targeted_productivity,2
team,2
smv,2
day,2
idle_men,2
over_time,2
incentive,2


# Part A — Scikit-learn

The assignment asks us to:

- handle missing values
- encode categorical features
- scale numerical features
- train Linear Regression
- train Logistic Regression
- record training and prediction time
- calculate the required metrics

The regression target is `actual_productivity`.

For classification we create:

\[
MeetsTarget =
\begin{cases}
1 & actual\_productivity \ge targeted\_productivity \\
0 & otherwise
\end{cases}
\]

`actual_productivity` is then removed from the classification inputs so that the answer is not leaked into the model.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Clean column names

df.columns = df.columns.astype(str).str.strip()

print("Columns in dataset:")
print(df.columns.tolist())


# 2. Remove corrupted / repeated-header rows

# Remove rows where the CSV header appears again as data.
df = df[df["department"].astype(str).str.strip() != "department"].copy()

# Remove obvious separator / git-conflict rows.
df = df[~df["date"].astype(str).str.startswith("=======")].copy()

df = df[~df["date"].astype(str).str.startswith(">>>>>>>")].copy()


# 3. Strip whitespace from categorical columns

categorical_columns = [
    "quarter",
    "department",
    "day"
]

for col in categorical_columns:
    df[col] = df[col].astype("string").str.strip()


# 4. Convert numerical columns to numeric

numerical_columns = [
    "team",
    "targeted_productivity",
    "smv",
    "wip",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers",
    "actual_productivity"
]

for col in numerical_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# 5. Display cleaned dataset information

print("\nCleaned dataset shape:")
print(df.shape)

print("\nCleaned columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
display(
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("Missing Values")
)

# 6. Regression target

y_reg = df["actual_productivity"].copy()


# 7. Classification target

y_cls = (
    df["actual_productivity"]
    >= df["targeted_productivity"]
).astype(int)

# 8. Create feature dataframe

X = df.drop(
    columns=[
        "actual_productivity",
        "date"
    ],
    errors="ignore"
).copy()

print("\nFeature columns:")
print(X.columns.tolist())

print("\nRegression target:")
print(y_reg.name)

print("\nClassification target:")
print("MeetsTarget")

print("\nClass distribution:")
display(
    y_cls
    .value_counts()
    .rename_axis("MeetsTarget")
    .to_frame("Count")
)

Columns in dataset:
['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'actual_productivity']

Cleaned dataset shape:
(2394, 15)

Cleaned columns:
['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'actual_productivity']

Data types:
date                         str
quarter                   string
department                string
day                       string
team                       int64
targeted_productivity    float64
smv                      float64
wip                      float64
over_time                  int64
incentive                  int64
idle_time                float64
idle_men                   int64
no_of_style_change         int64
no_of_workers            float64
actual_productivity      float64
dtype: object

Fi

,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382



Missing values:


,Missing Values
wip,1012
quarter,0
date,0
day,0
team,0
targeted_productivity,0
department,0
smv,0
over_time,0
incentive,0



Feature columns:
['quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']

Regression target:
actual_productivity

Classification target:
MeetsTarget

Class distribution:


,Count
MeetsTarget,
1,1750
0,644


## 3. Fixed train/test split


In [12]:
all_indices = np.arange(len(X))

train_idx, test_idx = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)

pd.DataFrame({
    "index": np.concatenate([train_idx, test_idx]),
    "split": (
        ["train"] * len(train_idx)
        + ["test"] * len(test_idx)
    ),
}).to_csv(SPLIT_PATH, index=False)

print("Fixed split created:")
print("Training rows:", len(train_idx))
print("Testing rows :", len(test_idx))
print("Saved to     :", SPLIT_PATH.resolve())


Fixed split created:
Training rows: 1915
Testing rows : 479
Saved to     : D:\College work\PG\Sem 1\ML\Lec\202618061_VedantKapur_DS605\202618061_Lab_5\split_indices.csv


## 4. Scikit-learn preprocessing


In [13]:
categorical = ["quarter", "department", "day"]
categorical = [c for c in categorical if c in X.columns]

numeric = [c for c in X.columns if c not in categorical]

print("Categorical columns:", categorical)
print("Numerical columns :", numeric)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", categorical_pipe, categorical),
])


Categorical columns: ['quarter', 'department', 'day']
Numerical columns : ['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']


## 5. Scikit-learn Linear Regression


In [14]:
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_reg_train = y_reg.iloc[train_idx]
y_reg_test = y_reg.iloc[test_idx]

regression_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LinearRegression()),
])

start = time.perf_counter()
regression_pipeline.fit(X_train, y_reg_train)
reg_train_time = time.perf_counter() - start

start = time.perf_counter()
reg_pred = regression_pipeline.predict(X_test)
reg_predict_time = time.perf_counter() - start

sklearn_regression_result = {
    "Model": "Scikit-learn LinearRegression",
    "MAE": mean_absolute_error(y_reg_test, reg_pred),
    "RMSE": np.sqrt(mean_squared_error(y_reg_test, reg_pred)),
    "R2": r2_score(y_reg_test, reg_pred),
    "Training_Time_s": reg_train_time,
    "Prediction_Time_s": reg_predict_time,
}

display(pd.DataFrame([sklearn_regression_result]))


,Model,MAE,RMSE,R2,Training_Time_s,Prediction_Time_s
0,Scikit-learn LinearRegression,0.110589,0.152659,0.216124,0.040634,0.004223


## 6. Scikit-learn Logistic Regression

We evaluate:

- Accuracy
- Precision
- Recall
- F1-score

In [15]:
y_cls_train = y_cls.iloc[train_idx]
y_cls_test = y_cls.iloc[test_idx]

classification_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=3000)),
])

start = time.perf_counter()
classification_pipeline.fit(X_train, y_cls_train)
cls_train_time = time.perf_counter() - start

start = time.perf_counter()
cls_pred = classification_pipeline.predict(X_test)
cls_predict_time = time.perf_counter() - start

sklearn_classification_result = {
    "Model": "Scikit-learn LogisticRegression",
    "Accuracy": accuracy_score(y_cls_test, cls_pred),
    "Precision": precision_score(y_cls_test, cls_pred, zero_division=0),
    "Recall": recall_score(y_cls_test, cls_pred, zero_division=0),
    "F1": f1_score(y_cls_test, cls_pred, zero_division=0),
    "Training_Time_s": cls_train_time,
    "Prediction_Time_s": cls_predict_time,
}

display(pd.DataFrame([sklearn_classification_result]))


,Model,Accuracy,Precision,Recall,F1,Training_Time_s,Prediction_Time_s
0,Scikit-learn LogisticRegression,0.74739,0.758621,0.953757,0.84507,0.025011,0.004172


# Part B — From Scratch with NumPy and Pandas



In [16]:
# Recreate X and targets from the already-loaded Pandas dataframe.
# No sklearn is used in this section.

X_manual = df.drop(columns=["actual_productivity", "date"], errors="ignore").copy()

y_reg_manual = df["actual_productivity"].to_numpy(dtype=float)

y_cls_manual = (
    df["actual_productivity"] >= df["targeted_productivity"]
).astype(int).to_numpy()

X_manual_train = X_manual.iloc[train_idx].copy()
X_manual_test = X_manual.iloc[test_idx].copy()

print("Manual train shape:", X_manual_train.shape)
print("Manual test shape :", X_manual_test.shape)


Manual train shape: (1915, 13)
Manual test shape : (479, 13)


In [17]:
def fit_preprocessor_manual(X_train):
    categorical = [c for c in ["quarter", "department", "day"] if c in X_train.columns]
    numeric = [c for c in X_train.columns if c not in categorical]

    params = {
        "categorical": categorical,
        "numeric": numeric,
        "medians": {},
        "means": {},
        "stds": {},
        "categories": {},
    }

    # Learn numeric parameters from TRAIN ONLY.
    for col in numeric:
        values = pd.to_numeric(X_train[col], errors="coerce")
        median = float(values.median())
        values = values.fillna(median)

        mean = float(values.mean())
        std = float(values.std(ddof=0))

        if std == 0 or not np.isfinite(std):
            std = 1.0

        params["medians"][col] = median
        params["means"][col] = mean
        params["stds"][col] = std

    # Learn categorical levels from TRAIN ONLY.
    for col in categorical:
        values = X_train[col].astype(str).str.strip()

        # Treat textual NaN as missing.
        values = values.replace("nan", np.nan)

        mode = values.mode()
        fill_value = str(mode.iloc[0]) if not mode.empty else "missing"

        values = values.fillna(fill_value)
        levels = sorted(values.unique().tolist())

        params["categories"][col] = {
            "fill": fill_value,
            "levels": levels,
        }

    return params


def transform_manual(X, params):
    blocks = []

    # Numerical features: impute + standardize.
    for col in params["numeric"]:
        values = pd.to_numeric(X[col], errors="coerce")
        values = values.fillna(params["medians"][col]).to_numpy(dtype=float)

        scaled = (
            values - params["means"][col]
        ) / params["stds"][col]

        blocks.append(scaled.reshape(-1, 1))

    # Categorical features: manual one-hot encoding.
    for col in params["categorical"]:
        info = params["categories"][col]

        values = X[col].astype(str).str.strip()
        values = values.replace("nan", np.nan)
        values = values.fillna(info["fill"])

        encoded = np.column_stack([
            (values.to_numpy() == category).astype(float)
            for category in info["levels"]
        ])

        blocks.append(encoded)

    return np.hstack(blocks).astype(float)


manual_params = fit_preprocessor_manual(X_manual_train)

X_manual_train_processed = transform_manual(X_manual_train, manual_params)
X_manual_test_processed = transform_manual(X_manual_test, manual_params)

print("Processed train shape:", X_manual_train_processed.shape)
print("Processed test shape :", X_manual_test_processed.shape)


Processed train shape: (1915, 23)
Processed test shape : (479, 23)


## 7. Manual Linear Regression


In [18]:
def add_intercept(X):
    return np.column_stack([np.ones(X.shape[0]), X])


class ManualLinearRegression:
    def __init__(self):
        self.beta = None

    def fit(self, X, y):
        Xb = add_intercept(X)
        self.beta = np.linalg.pinv(Xb) @ y
        return self

    def predict(self, X):
        if self.beta is None:
            raise ValueError("Model has not been fitted.")
        return add_intercept(X) @ self.beta


manual_reg_model = ManualLinearRegression()

start = time.perf_counter()
manual_reg_model.fit(
    X_manual_train_processed,
    y_reg_manual[train_idx],
)
manual_reg_train_time = time.perf_counter() - start

start = time.perf_counter()
manual_reg_pred = manual_reg_model.predict(X_manual_test_processed)
manual_reg_predict_time = time.perf_counter() - start


In [19]:
def manual_mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


def manual_rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def manual_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return float(1 - ss_res / ss_tot)


manual_regression_result = {
    "Model": "Manual LinearRegression",
    "MAE": manual_mae(y_reg_manual[test_idx], manual_reg_pred),
    "RMSE": manual_rmse(y_reg_manual[test_idx], manual_reg_pred),
    "R2": manual_r2(y_reg_manual[test_idx], manual_reg_pred),
    "Training_Time_s": manual_reg_train_time,
    "Prediction_Time_s": manual_reg_predict_time,
}

display(pd.DataFrame([manual_regression_result]))


,Model,MAE,RMSE,R2,Training_Time_s,Prediction_Time_s
0,Manual LinearRegression,0.110308,0.152764,0.215046,0.019148,0.00009


## 8. Manual Logistic Regression


In [20]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


class ManualLogisticRegression:
    def __init__(
        self,
        learning_rate=0.05,
        epochs=5000,
        l2=0.001,
        tolerance=1e-8,
        patience=100,
    ):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.l2 = l2
        self.tolerance = tolerance
        self.patience = patience
        self.weights = None
        self.loss_history = []

    def fit(self, X, y):
        Xb = add_intercept(X)
        n, p = Xb.shape

        self.weights = np.zeros(p, dtype=float)

        best_loss = np.inf
        stale = 0

        for _ in range(self.epochs):
            scores = Xb @ self.weights
            probabilities = sigmoid(scores)

            error = probabilities - y
            gradient = (Xb.T @ error) / n

            # L2 regularization, excluding intercept.
            reg = np.zeros_like(self.weights)
            reg[1:] = (self.l2 / n) * self.weights[1:]
            gradient += reg

            self.weights -= self.learning_rate * gradient

            p_safe = np.clip(probabilities, 1e-12, 1 - 1e-12)

            loss = -np.mean(
                y * np.log(p_safe)
                + (1 - y) * np.log(1 - p_safe)
            )

            loss += (
                self.l2 / (2 * n)
            ) * np.sum(self.weights[1:] ** 2)

            self.loss_history.append(loss)

            if best_loss - loss > self.tolerance:
                best_loss = loss
                stale = 0
            else:
                stale += 1

            if stale >= self.patience:
                break

        return self

    def predict_proba(self, X):
        return sigmoid(add_intercept(X) @ self.weights)

    def predict(self, X, threshold=0.5):
        return (
            self.predict_proba(X) >= threshold
        ).astype(int)


In [21]:
def manual_classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "Accuracy": float(accuracy),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
    }


manual_cls_model = ManualLogisticRegression(
    learning_rate=0.05,
    epochs=5000,
    l2=0.001,
)

start = time.perf_counter()
manual_cls_model.fit(
    X_manual_train_processed,
    y_cls_manual[train_idx],
)
manual_cls_train_time = time.perf_counter() - start

start = time.perf_counter()
manual_cls_pred = manual_cls_model.predict(
    X_manual_test_processed
)
manual_cls_predict_time = time.perf_counter() - start

manual_classification_result = {
    "Model": "Manual LogisticRegression",
    **manual_classification_metrics(
        y_cls_manual[test_idx],
        manual_cls_pred,
    ),
    "Training_Time_s": manual_cls_train_time,
    "Prediction_Time_s": manual_cls_predict_time,
}

display(pd.DataFrame([manual_classification_result]))


,Model,Accuracy,Precision,Recall,F1,Training_Time_s,Prediction_Time_s
0,Manual LogisticRegression,0.74739,0.758621,0.953757,0.84507,0.261198,0.000082


# Part C — Manual Optimization



In [22]:
# Create a validation split from the training portion only.
inner_indices = np.arange(len(train_idx))

inner_train_local, validation_local = train_test_split(
    inner_indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    shuffle=True,
)

tune_train_idx = train_idx[inner_train_local]
validation_idx = train_idx[validation_local]

X_tune_train_raw = X_manual.iloc[tune_train_idx]
X_validation_raw = X_manual.iloc[validation_idx]

tune_params = fit_preprocessor_manual(X_tune_train_raw)

X_tune_train = transform_manual(X_tune_train_raw, tune_params)
X_validation = transform_manual(X_validation_raw, tune_params)

y_tune = y_cls_manual[tune_train_idx]
y_validation = y_cls_manual[validation_idx]

learning_rates = [0.005, 0.01, 0.03, 0.05, 0.1]
l2_values = [0.0, 0.0001, 0.001, 0.01, 0.1]

tuning_results = []

for lr in learning_rates:
    for l2_value in l2_values:
        candidate = ManualLogisticRegression(
            learning_rate=lr,
            epochs=5000,
            l2=l2_value,
        )

        candidate.fit(X_tune_train, y_tune)

        validation_pred = candidate.predict(X_validation)
        metrics = manual_classification_metrics(
            y_validation,
            validation_pred,
        )

        tuning_results.append({
            "learning_rate": lr,
            "l2": l2_value,
            **metrics,
            "epochs_used": len(candidate.loss_history),
        })

tuning_df = pd.DataFrame(tuning_results)
tuning_df = tuning_df.sort_values(
    by="F1",
    ascending=False,
).reset_index(drop=True)

display(tuning_df.head(10))


,learning_rate,l2,Accuracy,Precision,Recall,F1,epochs_used
0,0.03,0.0000,0.772846,0.770588,0.96679,0.857610,5000
1,0.03,0.0100,0.772846,0.770588,0.96679,0.857610,5000
2,0.03,0.0010,0.772846,0.770588,0.96679,0.857610,5000
3,0.03,0.0001,0.772846,0.770588,0.96679,0.857610,5000
4,0.03,0.1000,0.772846,0.770588,0.96679,0.857610,5000
5,0.05,0.1000,0.767624,0.769231,0.95941,0.853859,5000
6,0.05,0.0001,0.762402,0.767857,0.95203,0.850082,5000
7,0.05,0.0000,0.762402,0.767857,0.95203,0.850082,5000
8,0.05,0.0100,0.762402,0.767857,0.95203,0.850082,5000
9,0.05,0.0010,0.762402,0.767857,0.95203,0.850082,5000


## 9. Train the optimized manual model


In [23]:
best_lr = float(tuning_df.loc[0, "learning_rate"])
best_l2 = float(tuning_df.loc[0, "l2"])

# Fit preprocessing on ALL training rows for the final model.
final_params = fit_preprocessor_manual(X_manual_train)

X_final_train = transform_manual(X_manual_train, final_params)
X_final_test = transform_manual(X_manual_test, final_params)

optimized_model = ManualLogisticRegression(
    learning_rate=best_lr,
    epochs=5000,
    l2=best_l2,
)

start = time.perf_counter()
optimized_model.fit(
    X_final_train,
    y_cls_manual[train_idx],
)
optimized_train_time = time.perf_counter() - start

start = time.perf_counter()
optimized_pred = optimized_model.predict(X_final_test)
optimized_predict_time = time.perf_counter() - start

optimized_classification_result = {
    "Model": "Optimized Manual LogisticRegression",
    **manual_classification_metrics(
        y_cls_manual[test_idx],
        optimized_pred,
    ),
    "Training_Time_s": optimized_train_time,
    "Prediction_Time_s": optimized_predict_time,
    "Learning_Rate": best_lr,
    "L2": best_l2,
}

display(pd.DataFrame([optimized_classification_result]))


,Model,Accuracy,Precision,Recall,F1,Training_Time_s,Prediction_Time_s,Learning_Rate,L2
0,Optimized Manual LogisticRegression,0.74739,0.756264,0.959538,0.84586,0.255066,0.000074,0.03,0.0


# 10. Final comparison tables


In [24]:
regression_comparison = pd.DataFrame([
    sklearn_regression_result,
    manual_regression_result,
])

classification_comparison = pd.DataFrame([
    sklearn_classification_result,
    manual_classification_result,
    optimized_classification_result,
])

print("REGRESSION COMPARISON")
display(regression_comparison)

print("CLASSIFICATION COMPARISON")
display(classification_comparison)

regression_comparison.to_csv(
    "regression_comparison.csv",
    index=False,
)

classification_comparison.to_csv(
    "classification_comparison.csv",
    index=False,
)

pd.DataFrame([sklearn_regression_result]).to_csv(
    "sklearn_regression_results.csv",
    index=False,
)

pd.DataFrame([sklearn_classification_result]).to_csv(
    "sklearn_classification_results.csv",
    index=False,
)

pd.DataFrame([manual_regression_result]).to_csv(
    "manual_regression_results.csv",
    index=False,
)

pd.DataFrame([manual_classification_result]).to_csv(
    "manual_classification_results.csv",
    index=False,
)

print("Result CSV files saved.")


REGRESSION COMPARISON


,Model,MAE,RMSE,R2,Training_Time_s,Prediction_Time_s
0,Scikit-learn LinearRegression,0.110589,0.152659,0.216124,0.040634,0.004223
1,Manual LinearRegression,0.110308,0.152764,0.215046,0.019148,0.000090


CLASSIFICATION COMPARISON


,Model,Accuracy,Precision,Recall,F1,Training_Time_s,Prediction_Time_s,Learning_Rate,L2
0,Scikit-learn LogisticRegression,0.74739,0.758621,0.953757,0.84507,0.025011,0.004172,NaN,NaN
1,Manual LogisticRegression,0.74739,0.758621,0.953757,0.84507,0.261198,0.000082,NaN,NaN
2,Optimized Manual LogisticRegression,0.74739,0.756264,0.959538,0.84586,0.255066,0.000074,0.03,0.0


Result CSV files saved.
